<a href="https://colab.research.google.com/github/hursoo/big_k-modern_1/blob/main/gb_011_scraping(big1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.필요한 도구와 라이브러리 설치

In [1]:
# 구글 드라이브 마운트

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 경로 지정

file_path = '/content/drive/MyDrive/big_km_history01/'  # 맨 마지막에 '/' 추가

In [3]:
import ssl
import pandas as pd
from urllib.request import urlopen
from bs4 import BeautifulSoup

# 2.기사정보 다운로드

사전 작업
- 국편 한국사데이터베이스/개벽에서 기사정보 다운로드 = "근현대잡지자료_20250315172708.txt"

In [4]:
gisa_info = pd.read_csv(file_path + 'data/근현대잡지자료_20250315172708.txt', sep = '^', encoding='utf-8')
gisa_info

,자료ID,잡지명,발행일,기사제목,필자,기사형태,URL
0,ma_013_0010,개벽 제1호,1920-06-25,개벽 제1호,NaN,NaN,https://db.history.go.kr/id/ma_013_0010
1,ma_013_0010_0001,개벽 제1호,NaN,謝告,NaN,사고·편집후기,https://db.history.go.kr/id/ma_013_0010_0001
2,ma_013_0010_0010,개벽 제1호,NaN,권두시,NaN,시,https://db.history.go.kr/id/ma_013_0010_0010
3,ma_013_0010_0020,개벽 제1호,NaN,創刊辭,NaN,사고·편집후기,https://db.history.go.kr/id/ma_013_0010_0020
4,ma_013_0010_0030,개벽 제1호,NaN,世界를 알라,NaN,논설,https://db.history.go.kr/id/ma_013_0010_0030
...,...,...,...,...,...,...,...
2534,ma_013_0750_0450,개벽 신간 제4호,NaN,中篇 巨人(2),金東仁,소설,https://db.history.go.kr/id/ma_013_0750_0450
2535,ma_013_0750_0460,개벽 신간 제4호,NaN,脫線,朴鄕民,희곡·시나리오,https://db.history.go.kr/id/ma_013_0750_0460
2536,ma_013_0750_0461,개벽 신간 제4호,NaN,編輯餘墨,NaN,사고·편집후기,https://db.history.go.kr/id/ma_013_0750_0461
2537,ma_013_0750_0462,개벽 신간 제4호,NaN,謹告,NaN,사고·편집후기,https://db.history.go.kr/id/ma_013_0750_0462


# 3.기사정보로부터 주요논설 선별
- 오프라인(코드 바깥) 작업
- 2000여개 기사를 보고 필요한 '주요논설'을 선별

# 4.주요논설 334개 목록파일 불러옴

In [5]:
# 주요 논설 id
r334_info = pd.read_excel(file_path + 'data/gb_data_2.1.xlsx', sheet_name = 'ron_info')
r334_info

,r_id,r_id_raw,title,writer,gisa_class,date,url,year
0,1,ma_013_0010_0020,創刊辭,NaN,사고·편집후기,1920-06-25,http://db.history.go.kr/id/ma_013_0010_0020,1920
1,2,ma_013_0010_0030,世界를 알라,NaN,논설,1920-06-25,http://db.history.go.kr/id/ma_013_0010_0030,1920
2,3,ma_013_0010_0060,時急히 解決할 朝鮮의 二大問題,朴達成,논설,1920-06-25,http://db.history.go.kr/id/ma_013_0010_0060,1920
3,4,ma_013_0020_0020,"世界 三大 問題의 波及과 朝鮮人의 覺悟如何, 社說",NaN,논설,1920-07-25,http://db.history.go.kr/id/ma_013_0020_0020,1920
4,5,ma_013_0020_0040,最近 朝鮮社會運動의 二三,李敦化,논설,1920-07-25,http://db.history.go.kr/id/ma_013_0020_0040,1920
...,...,...,...,...,...,...,...,...
329,330,ma_013_0710_0010,작년 이때를 생각하면서,NaN,사고·편집후기,1926-08-01,http://db.history.go.kr/id/ma_013_0710_0010,1926
330,331,ma_013_0710_0020,"新興政治運動의 性質, 全的 運動과 部分的 運動의 分野",裵成龍,논설,1926-08-01,http://db.history.go.kr/id/ma_013_0710_0020,1926
331,332,ma_013_0710_0030,"朝鮮의 人口論(二), 第二編 動態人口",李順鐸,논설,1926-08-01,http://db.history.go.kr/id/ma_013_0710_0030,1926
332,333,ma_013_0710_0040,着目할 日本의 移民政策,NaN,논설,1926-08-01,http://db.history.go.kr/id/ma_013_0710_0040,1926


In [6]:
urls = r334_info['url'].tolist()
urls[:3]

['http://db.history.go.kr/id/ma_013_0010_0020',
 'http://db.history.go.kr/id/ma_013_0010_0030',
 'http://db.history.go.kr/id/ma_013_0010_0060']

# 5.주요 논설 원문 스크래핑
- 기사정보와 주요논설 목록을 활용.
- url 주소를 이용하여 웹 페이지에 접속하고, BeautifulSoup을 사용하여 원하는 부분의 데이터를 추출

In [7]:
def get_contents(urls, n):
    results = []

    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    for url in urls[:n]:
        webpage = urlopen(url, context=ctx)
        r_id = url[-16:]

        bsobj = BeautifulSoup(webpage.read(), 'html.parser')
        List1 = bsobj.find_all('div', {'id': 'cont_view'})

        for z in List1:
            z1 = z.get_text('\n', strip=True)
            results.append([r_id, z1])

    result_df = pd.DataFrame(results, columns=['r_id', 'content'])
    return result_df

In [8]:
contents_df = get_contents(urls, 10)
contents_df

,r_id,content
0,ma_013_0010_0020,創刊辭\n소리-있어 넓히 世界에 傳하니 온 世界 모든 人類-이에 應하야 부르짖기를 ...
1,ma_013_0010_0030,世界를 알라\n一\n사람은 天使도 안이며 野獸도 안이오 오즉 사람일 뿐이로다. 이만...
2,ma_013_0010_0060,時急히 解決할 朝鮮의 二大問題\n朴達成\n남은 汽車를 타고 千里에 달아나는데 우리는...
3,ma_013_0020_0020,"世界 三大 問題의 波及과 朝鮮人의 覺悟如何, 社說\n1. 三大 問題는 그를 吸收하고..."
4,ma_013_0020_0040,"最近 朝鮮社會運動의 二三\n李敦化\n朝鮮敎育會, 朝鮮學生大會, 朝鮮勞働共濟會의 趣旨..."
5,ma_013_0020_0050,"急激히 向上되는 朝鮮靑年의 思想界, 可賀할 朝鮮靑年의 知識熱\n朴達成\n無私한 태양..."
6,ma_013_0030_0020,吾人의 新紀元을 宣言하노라\n吾人은 均히 自國 歷史의 長遠을 자랑하고 自家譜乘의 久...
7,ma_013_0030_0030,新時代와 新人物\n一. 改造라 함은 人類의 靈力이 自然의 上에 作用함을 이름이라\n...
8,ma_013_0030_0050,家族制度의 側面觀\n滄海居士\n一. 家族制度의 變遷\n家族制度에도 美點이 업는 바 ...
9,ma_013_0030_0070,朝鮮女子의 今後行路\n妙香山人\n누의님 悾惚한 이 歲月이 일만흔 우리에게 더욱 悾惚...


In [9]:
# 본문 내용 확인 (첫 행)
#contents_5_df.loc[3,'Content']
#print(contents_df.loc[3,'content'])

# 6.데이터 저장

In [10]:
r334_info1 = r334_info.drop('r_id', axis=1) # 두 df의 중복열 중 미리 한쪽을 제거
combi_df = pd.merge(r334_info1, contents_df, left_on='r_id_raw', right_on = 'r_id', how='inner')
combi_df1 = combi_df[['r_id', 'r_id_raw', 'title', 'writer', 'gisa_class', 'date', 'url', 'year', 'content']]
combi_df1

,r_id,r_id_raw,title,writer,gisa_class,date,url,year,content
0,ma_013_0010_0020,ma_013_0010_0020,創刊辭,NaN,사고·편집후기,1920-06-25,http://db.history.go.kr/id/ma_013_0010_0020,1920,創刊辭\n소리-있어 넓히 世界에 傳하니 온 世界 모든 人類-이에 應하야 부르짖기를 ...
1,ma_013_0010_0030,ma_013_0010_0030,世界를 알라,NaN,논설,1920-06-25,http://db.history.go.kr/id/ma_013_0010_0030,1920,世界를 알라\n一\n사람은 天使도 안이며 野獸도 안이오 오즉 사람일 뿐이로다. 이만...
2,ma_013_0010_0060,ma_013_0010_0060,時急히 解決할 朝鮮의 二大問題,朴達成,논설,1920-06-25,http://db.history.go.kr/id/ma_013_0010_0060,1920,時急히 解決할 朝鮮의 二大問題\n朴達成\n남은 汽車를 타고 千里에 달아나는데 우리는...
3,ma_013_0020_0020,ma_013_0020_0020,"世界 三大 問題의 波及과 朝鮮人의 覺悟如何, 社說",NaN,논설,1920-07-25,http://db.history.go.kr/id/ma_013_0020_0020,1920,"世界 三大 問題의 波及과 朝鮮人의 覺悟如何, 社說\n1. 三大 問題는 그를 吸收하고..."
4,ma_013_0020_0040,ma_013_0020_0040,最近 朝鮮社會運動의 二三,李敦化,논설,1920-07-25,http://db.history.go.kr/id/ma_013_0020_0040,1920,"最近 朝鮮社會運動의 二三\n李敦化\n朝鮮敎育會, 朝鮮學生大會, 朝鮮勞働共濟會의 趣旨..."
5,ma_013_0020_0050,ma_013_0020_0050,"急激히 向上되는 朝鮮靑年의 思想界, 可賀할 朝鮮靑年의 知識熱",朴達成,논설,1920-07-25,http://db.history.go.kr/id/ma_013_0020_0050,1920,"急激히 向上되는 朝鮮靑年의 思想界, 可賀할 朝鮮靑年의 知識熱\n朴達成\n無私한 태양..."
6,ma_013_0030_0020,ma_013_0030_0020,吾人의 新紀元을 宣言하노라,NaN,논설,1920-08-25,http://db.history.go.kr/id/ma_013_0030_0020,1920,吾人의 新紀元을 宣言하노라\n吾人은 均히 自國 歷史의 長遠을 자랑하고 自家譜乘의 久...
7,ma_013_0030_0030,ma_013_0030_0030,新時代와 新人物,NaN,논설,1920-08-25,http://db.history.go.kr/id/ma_013_0030_0030,1920,新時代와 新人物\n一. 改造라 함은 人類의 靈力이 自然의 上에 作用함을 이름이라\n...
8,ma_013_0030_0050,ma_013_0030_0050,家族制度의 側面觀,滄海居士,논설,1920-08-25,http://db.history.go.kr/id/ma_013_0030_0050,1920,家族制度의 側面觀\n滄海居士\n一. 家族制度의 變遷\n家族制度에도 美點이 업는 바 ...
9,ma_013_0030_0070,ma_013_0030_0070,朝鮮女子의 今後行路,妙香山人,논설,1920-08-25,http://db.history.go.kr/id/ma_013_0030_0070,1920,朝鮮女子의 今後行路\n妙香山人\n누의님 悾惚한 이 歲月이 일만흔 우리에게 더욱 悾惚...


In [11]:
# 저장
combi_df1.to_excel(file_path + 'result/ron10_data.xlsx', index=False)  # 인덱스를 저장하지 않도록 설정

# The End of Note